# Домашнее задание 2. Выпуклость и множители Лагранжа

**Выдано:** 23 сентября 2026 (обновлено 29 сентября). **Срок сдачи:** 7 октября 2026 (до начала занятия). **Максимум:** 10 баллов (+1 бонус).

**Как сдавать.** Заполните этот ноутбук (ответы на теоретические вопросы — в markdown-ячейках, формулы в $\LaTeX$; код — в code-ячейках), выполните целиком (`Kernel → Restart Kernel and Run All Cells`) и сдайте через pull request: файл `submissions/hw02/<фамилия>.ipynb`, инструкция — [`CONTRIBUTING.md`](../CONTRIBUTING.md). Задачи можно обсуждать, но текст и код пишутся самостоятельно.

**Материал:** лекция 2 ([конспект](../lectures/lecture02/lecture02.md), [демо](../lectures/lecture02/demo02.ipynb)) — операции, сохраняющие выпуклость (раздел 5), чек-лист выпуклой задачи (раздел 6), условие оптимальности (раздел 7); лекция 3 ([конспект](../lectures/lecture03/lecture03.md)) — баланс сил $\nabla f = \mu \nabla h$, касание $\nabla f = \lambda \nabla g$, функция Лагранжа и множитель как цена (разделы 4–8). Ничего сверх лекций не требуется. Ориентировочное время — 4–5 часов.

**Сложность** отмечена звёздочками: ★ — разминка, ★★ — стандартная задача, ★★★ — нужно подумать. Большая часть задания решается на бумаге; код нужен в задаче 3 и для коротких проверок.

| Задача | Баллы | О чём |
|--------|-------|-------|
| 1. Выпуклость | 3 | множества, критерий второго порядка, операции |
| 2. Три функции | 2 | функция Розенброка, log-sum-exp, наибольшее собственное число |
| 3. Выпуклая и невыпуклая подгонка | 2 | синус с неизвестной и известной частотой, мультистарт |
| 4. Множители Лагранжа | 3 | проекция на полупространство, собственные числа как касание, «водозаполнение» |
| Бонус | +1 | максимум энтропии и распределение Больцмана |

Запуск: `uv sync` в корне репозитория, затем `uv run jupyter lab`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize

rng = np.random.default_rng(2026)   # для ваших собственных случайных проверок; данные задач генерируются отдельно

## Задача 1. Выпуклость (3 балла)

По 1 баллу за пункт. В каждом пункте нужен либо аргумент «почему выпукло» — определение, критерий первого/второго порядка или операция из раздела 5 конспекта, — либо контрпример: две точки и вылезающий наружу отрезок, хорда под графиком или точка, где гессиан не полуопределён. Численная проверка приветствуется, но доказательством не считается (демо, часть a).

**(а) Множества** ★. Выпуклы ли:

1. $\{x \in \mathbb{R}^2 : x_2 \ge x_1^2\}$;
2. $\{x \in \mathbb{R}^n : \Vert Ax - b\Vert_2 \le 1\}$ при произвольных $A \in \mathbb{R}^{m \times n}$, $b \in \mathbb{R}^m$;
3. $\{x \in \mathbb{R}^2 : \Vert x\Vert_2 \ge 1\}$?

**(б) Критерий второго порядка** ★.

1. При каких $a \in \mathbb{R}$ функция $f(x) = x_1^2 + a\,x_1 x_2 + x_2^2$ выпукла на $\mathbb{R}^2$? При каких — строго выпукла? Проверьте ответ через `np.linalg.eigvalsh` для нескольких значений $a$.
2. Выпукла ли $g(x) = x_1^2 x_2^2$ на $\mathbb{R}^2$?

**(в) Операции** ★★. Докажите выпуклость или объясните, почему правила раздела 5 не применимы, и приведите контрпример:

1. $f(x) = \Vert Ax - b\Vert_2 + \lambda\,\Vert x\Vert_\infty$, $\lambda \ge 0$;
2. $f(x) = \max_{i} \big(a_i^\top x - b_i\big)^2$;
3. $f(x) = e^{-\Vert x\Vert_2^2}$.

Какие из трёх функций гладкие?

**(а)**

**Ответ:** _(ваш текст; формулы — в $\LaTeX$)_

**(б)**

**Ответ:** _(ваш текст; формулы — в $\LaTeX$)_

In [ ]:
# ваш код


**(в)**

**Ответ:** _(ваш текст; формулы — в $\LaTeX$)_

## Задача 2. Три функции (2 балла)

**(а) Функция Розенброка** ★ (0.5 балла). $f(x) = (1 - x_1)^2 + 100\,(x_2 - x_1^2)^2$ — «банановая долина», на которой в части II будем испытывать все методы.

1. Докажите, что $f$ не выпукла: предъявите хорду, лежащую под графиком, или точку, где гессиан не полуопределён.
2. Найдите множество точек, где $\nabla^2 f(x) \succeq 0$. Как оно расположено относительно дна долины $x_2 = x_1^2$?
3. Найдите все стационарные точки $f$. Почему у этой невыпуклой функции единственный локальный минимум оказался глобальным и не противоречит ли это лекции 2?

**(б) Гладкий максимум: log-sum-exp** ★★ (0.75 балла). $\operatorname{lse}(x) = \log \sum_{i=1}^n e^{x_i}$, $x \in \mathbb{R}^n$.

1. Докажите, что $\max_i x_i \le \operatorname{lse}(x) \le \max_i x_i + \log n$.
2. Для $t > 0$ положим $\operatorname{lse}_t(x) = \tfrac1t \operatorname{lse}(t x)$. Покажите, что $\operatorname{lse}_t(x) \to \max_i x_i$ при $t \to \infty$, и оцените погрешность.
3. Вычислите градиент и гессиан $\operatorname{lse}$ и докажите, что $\operatorname{lse}$ выпукла. Подсказка: обозначьте $p_i = e^{x_i} / \sum_j e^{x_j}$ и сведите $v^\top \nabla^2 \operatorname{lse}(x)\, v \ge 0$ к неравенству Коши–Буняковского (или к неотрицательности дисперсии). Строго ли она выпукла?
4. Проверьте оценку из пункта 2 численно для случайного $x \in \mathbb{R}^{10}$ с координатами порядка $10$ и $t = 1, 10, 100$. Посчитайте $\operatorname{lse}$ «в лоб», `np.log(np.sum(np.exp(t * x))) / t`, и через `scipy.special.logsumexp`. Что ломается при $t = 100$ и как `logsumexp` этого избегает?

Зачем это нужно: $\max$ выпукл, но негладок, а $\operatorname{lse}_t$ — его гладкая выпуклая замена сверху. Так, например, сглаживают чебышёвскую подгонку из бонуса ДЗ 1.

**(в) Наибольшее собственное число** ★★★ (0.75 балла). $\mathbb{S}^n$ — пространство симметричных матриц $n \times n$, у $X \in \mathbb{S}^n$ собственные числа $\lambda_1(X) \ge \dots \ge \lambda_n(X)$.

1. Докажите, что $\lambda_1(X) = \max_{\Vert v\Vert = 1} v^\top X v$, и выведите отсюда, что $\lambda_1$ — выпуклая функция на $\mathbb{S}^n$, а $\lambda_n$ — вогнутая. Подсказка: при фиксированном $v$ отображение $X \mapsto v^\top X v$ линейно по $X$.
2. Выпукло ли множество положительно полуопределённых матриц $\{X \in \mathbb{S}^n : X \succeq 0\}$? А множество $\{X : \lambda_1(X) \le 1\}$?
3. Выпукла или вогнута на $\mathbb{S}^3$ функция $\lambda_2(X)$ — второе по величине собственное число? Достаточно диагональных матриц.

**(а)**

**Ответ:** _(ваш текст; формулы — в $\LaTeX$)_

In [ ]:
# ваш код


**(б)**

**Ответ:** _(ваш текст; формулы — в $\LaTeX$)_

In [ ]:
# ваш код


**(в)**

**Ответ:** _(ваш текст; формулы — в $\LaTeX$)_

In [ ]:
# ваш код


## Задача 3. Выпуклая и невыпуклая подгонка ★★ (2 балла)

Пункт 1 — 0.5 балла, пункты 2 и 3 — по 0.75.

Ниже сгенерированы измерения $y_i = 2\sin(3t_i + 0.7) + \varepsilon_i$, $\varepsilon_i \sim \mathcal{N}(0, 0.2^2)$, $i = 1, \dots, 60$. Один и тот же сигнал описываем двумя моделями:

- **A** (задача 1(г) ДЗ 1): $\varphi_A(t; x) = x_1 \sin(x_2 t + x_3)$, $x \in \mathbb{R}^3$ — амплитуда, частота и фаза неизвестны;
- **B** (частота известна, $\omega = 3$): $\varphi_B(t; x) = x_1 \sin\omega t + x_2 \cos\omega t$, $x \in \mathbb{R}^2$.

Цель в обоих случаях — сумма квадратов невязок $f(x) = \tfrac12\sum_i \big(\varphi(t_i; x) - y_i\big)^2$.

1. Какого класса каждая задача и выпукла ли она? Для B — докажите; для A — объясните, почему нет (подсказка: что происходит с $f$ при $x_3 \to x_3 + 2\pi$ и при $(x_1, x_3) \to (-x_1, x_3 + \pi)$?).
2. Запустите `minimize(..., method="BFGS")` для обеих моделей из 20 одинаковых случайных стартов `starts` (для B берите первые две координаты старта). Выведите итоговые значения $f$ по стартам и долю стартов, пришедших к наименьшему найденному значению; нарисуйте данные и подогнанные кривые для обеих моделей.
3. Объясните результат через главную теорему лекции. Как из решения B получить амплитуду и фазу для модели A? Почему наименьшее найденное $f$ у A чуть меньше, чем у B, и почему это не делает A «лучшей» задачей?

In [ ]:
data_rng = np.random.default_rng(2026)          # отдельный генератор: данные не зависят от вашего кода выше
N, omega = 60, 3.0
t = np.sort(data_rng.uniform(0, 2 * np.pi, N))
y = 2.0 * np.sin(3.0 * t + 0.7) + 0.2 * data_rng.standard_normal(N)
starts = data_rng.uniform(-5, 5, (20, 3))         # 20 общих стартов для обеих моделей

plt.figure(figsize=(6, 3))
plt.plot(t, y, "o", ms=4)
plt.xlabel("$t$"); plt.ylabel("$y$"); plt.title("Данные задачи 3")
plt.show()

**Класс и выпуклость (пункт 1)**

**Ответ:** _(ваш текст; формулы — в $\LaTeX$)_

**Мультистарт (пункт 2)**

In [ ]:
# ваш код


**Объяснение (пункт 3)**

**Ответ:** _(ваш текст; формулы — в $\LaTeX$)_

## Задача 4. Множители Лагранжа (3 балла)

По 1 баллу за пункт. Обозначения — как в лекции 3: задача $\min f(x)$ при $g(x) = 0$, $h(x) \ge 0$; $L = f - \lambda^\top g - \mu^\top h$; система ККТ — $\nabla_x L = 0$, допустимость, $\mu \ge 0$, $\mu_i h_i = 0$.

**(а) Проекция на полупространство: две дороги к одному ответу** ★. Пусть $\Omega = \{x \in \mathbb{R}^n : a^\top x \le b\}$, $a \ne 0$, и точка $p$ лежит вне $\Omega$: $a^\top p > b$. Задача проекции — $\min_{x \in \Omega} \tfrac12\Vert x - p\Vert^2$.

1. **Через условие лекции 2.** Из $\nabla f(x^\ast)^\top(y - x^\ast) \ge 0\ \ \forall y \in \Omega$ выведите
$$x^\ast = p - \frac{a^\top p - b}{\Vert a\Vert^2}\, a .$$
Подсказки: (i) почему $x^\ast$ не может лежать строго внутри $\Omega$; (ii) для любого $v$ с $a^\top v = 0$ точки $y = x^\ast \pm v$ допустимы — что даёт условие для них; (iii) коэффициент при $a$ найдите из $a^\top x^\ast = b$; (iv) убедитесь, что условие выполнено для всех $y \in \Omega$.
2. **Через баланс сил лекции 3.** Запишите $h(x) = b - a^\top x \ge 0$ и условие $\nabla f = \mu \nabla h$. Получите ту же формулу и значение $\mu^\ast$; почему $\mu^\ast \ge 0$ ровно тогда, когда $p \notin \Omega$? Как $\mu^\ast$ связано с расстоянием от $p$ до $\Omega$?
3. **Цена.** Найдите $f^\ast(b)$ и проверьте, что $df^\ast / db = -\mu^\ast$. Объясните знак.

**(б) Собственные числа как касание** ★★. $A \in \mathbb{S}^n$, задача $\min$ (или $\max$) $x^\top A x$ при $\Vert x\Vert^2 = 1$.

1. Запишите условие касания $\nabla f = \lambda \nabla g$ и покажите, что его решения — ровно единичные собственные векторы $A$, а множитель $\lambda$ равен собственному числу и значению $f$ в этой точке. Выведите, что $\min = \lambda_n(A)$, $\max = \lambda_1(A)$ (почему минимум и максимум вообще существуют?). Сравните с задачей 2(в).
2. Для $A = \operatorname{diag}(1, 2, 3)$ перечислите все точки, где выполнено условие касания. Покажите, что $\pm e_2$ — ни минимум, ни максимум: сдвиньтесь по сфере в сторону $e_1$ и в сторону $e_3$. Какой вывод о системе ККТ для невыпуклых задач? Выпукла ли эта задача?
3. Замените ограничение на $\Vert x\Vert^2 = r$, $r > 0$. Найдите $f^\ast_{\min}(r)$ и $df^\ast_{\min}/dr$ и сравните с $\lambda$.

**(в) Водозаполнение** ★★★. Мощность $P > 0$ распределяется между $n$ каналами связи с уровнями шума $\alpha_i > 0$; пропускная способность канала с мощностью $x_i$ равна $\log(\alpha_i + x_i)$ (с точностью до константы). Задача:
$$\max_x \sum_{i=1}^n \log(\alpha_i + x_i) \quad\text{при}\quad \sum_i x_i = P,\ \ x_i \ge 0 .$$
1. Приведите задачу к виду $\min f$ и докажите, что она выпукла.
2. Выпишите систему ККТ и выведите решение
$$x_i^\ast = \max(0,\ w - \alpha_i),$$
где «уровень воды» $w$ определяется уравнением $\sum_i \max(0, w - \alpha_i) = P$. Почему у этого уравнения ровно одно решение? Объясните название: на что похожа картинка «дно $\alpha_i$ + налитая вода $x_i$»? Какие каналы не получают ничего и какой у них множитель?
3. Для $\alpha = (0.4,\ 0.8,\ 1.2,\ 2.0,\ 3.0)$ и $P = 2$ найдите $w$, $x^\ast$, $\lambda^\ast$ и $\mu^\ast$ **руками**, затем проверьте кодом (`minimize` с `method="SLSQP"` или бисекция по $w$) и нарисуйте картинку «сосудов».
4. На сколько вырастет суммарная пропускная способность, если добавить немного мощности? Ответьте через множитель и проверьте конечной разностью.

In [ ]:
alpha = np.array([0.4, 0.8, 1.2, 2.0, 3.0])   # уровни шума в каналах, задача 4(в)
P = 2.0                                        # суммарная мощность

**(а)**

**Ответ:** _(ваш текст; формулы — в $\LaTeX$)_

**(б)**

**Ответ:** _(ваш текст; формулы — в $\LaTeX$)_

In [ ]:
# ваш код


**(в)**

**Ответ:** _(ваш текст; формулы — в $\LaTeX$)_

In [ ]:
# ваш код


## Бонус ★★★ (+1 балл). Максимум энтропии

Система может находиться в состояниях $i = 1, \dots, n$ с энергиями $E_i$. Какое распределение вероятностей $p$ наиболее «неопределённо» при известной средней энергии $\bar E$? Задача:
$$\max_p\ H(p) = -\sum_i p_i \log p_i \quad\text{при}\quad \sum_i p_i = 1,\ \ \sum_i p_i E_i = \bar E,\ \ p_i \ge 0$$
(с соглашением $0 \log 0 = 0$).

1. Докажите, что задача выпукла (в форме $\min -H$).
2. Без ограничения на энергию найдите решение через множитель Лагранжа.
3. С ограничением на энергию выпишите условия ККТ и покажите, что $p_i^\ast = e^{-\beta E_i} / Z$, $Z = \sum_j e^{-\beta E_j}$ (распределение Больцмана). Почему ограничения $p_i \ge 0$ оказываются неактивными? Как связан $\beta$ с множителями?
4. Для $E = (0, 1, 2, 3)$ и $\bar E = 1$ найдите $\beta$ численно (одно уравнение на одно число, `scipy.optimize.brentq`) и $p^\ast$. Почему $\beta > 0$? Проверьте конечной разностью, что $dH^\ast/d\bar E = \beta$ — это определение температуры из термодинамики, $\beta = 1/T$.

**Ответ:** _(ваш текст; формулы — в $\LaTeX$)_

In [ ]:
# ваш код


---

Перед сдачей: `Kernel → Restart Kernel and Run All Cells`, убедитесь, что ни одна ячейка не падает и все ответы заполнены. Файл — `submissions/hw02/<фамилия>.ipynb`, дальше по [инструкции](../CONTRIBUTING.md).